# NB 1.2 &mdash; Xarxes neuronals sobre dades reals: MNIST

**MP 5149** &mdash; Desenvolupament de components software per a sistemes d'aprenentatge automàtic

**UT5** &mdash; Introducció a les xarxes neuronals supervisades

---
### D'on venim

Al NB 1.1 vam entrenar un perceptró i un perceptró multicapa amb scikit-learn, i
vam entendre per què funcionen amb problemes que es podien **dibuixar**: quatre punts de
la XOR, pingüins amb dues mesures, dues llunes. Tot passava en un pla de dues
dimensions, i podíem veure la frontera.

Això era deliberat: per entendre una idea, convé que les dades siguin petites. Però
les xarxes neuronals no es van fer famoses per separar llunes.

### Què farem avui

Portarem l'MLP a un problema de veritat: **reconèixer dígits escrits a mà**.
És el conjunt **MNIST**: 70.000 imatges de 28 × 28 píxels, escrites per treballadors
de l'oficina del cens dels Estats Units i per estudiants de secundària, i publicades
el 1998 per Yann LeCun i col·laboradors. Durant anys va ser *la* prova de referència
de la visió per computador.

Canviar d'escala ens obligarà a introduir tres coses noves:

1. **De 2 classes a 10.** Una neurona de sortida per a cada dígit i la funció **softmax**.
2. **Mini-lots.** Amb 50.000 imatges d'entrenament ja no té sentit calcular el
   gradient sobre totes alhora.
3. **Validació durant l'entrenament**, per saber quan la xarxa comença a memoritzar.

I ens permetrà **veure amb els ulls** la idea central de la unitat: què han après les
neurones de la capa oculta.

### Preguntes que has de saber respondre en acabar

1. Per què una imatge de 28 × 28 píxels és una fila de 784 columnes, i què es perd pel camí?
2. Què fa la funció softmax i com sap `MLPClassifier` que ha de fer-la servir?
3. Per què s'entrena amb mini-lots i què és una iteració comparada amb una època?
4. Com es pot demostrar que la capa oculta ha après característiques útils?
5. Guanya sempre una xarxa neuronal a un bosc aleatori amb aquestes dades?
6. Què **no** entén un perceptró multicapa sobre una imatge?

> **Temps d'execució.** Aquest notebook entrena força models. A Colab, executar-lo
> sencer tarda uns quants minuts. Les cel·les que tarden més ho avisen.

## 0. Preparació

Les mateixes biblioteques que al NB 1.1. Afegim `io` i `urllib` de la biblioteca
estàndard de Python per descarregar el fitxer de dades, i `time` per mesurar quant
tarda cada entrenament.

Com al NB 1.1, no programarem cap xarxa: tots els models són de **scikit-learn**.

In [ ]:
import io
import time
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.metrics import log_loss
from sklearn.neural_network import MLPClassifier

# Copy of OpenML's mnist_784 kept in the MP 5149 repository (60,000 train + 10,000 test images)
URL_DATA = "https://raw.githubusercontent.com/pprohenspolitecnicllevant/desenvolupament-components-software/refs/heads/main/UT01-Xarxes_neuronals_supervisades/mnist/mnist.npz"

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

## 1. Les dades

### 1.1 Carregar el fitxer

Les dades són en un fitxer `.npz`, el format de NumPy per desar diversos arrays
comprimits en un sol fitxer. El descarreguem en memòria i l'obrim amb `np.load`.

In [ ]:
def load_npz_from_url(url):
    """Download a .npz file into memory and open it with NumPy."""
    with urllib.request.urlopen(url) as response:
        return np.load(io.BytesIO(response.read()))


start = time.perf_counter()
mnist = load_npz_from_url(URL_DATA)
print(f"Downloaded in {time.perf_counter() - start:.1f} s")
print("Arrays in the file:", list(mnist.keys()))

X_train_full, y_train_full = mnist["X_train"], mnist["y_train"]
X_test, y_test = mnist["X_test"], mnist["y_test"]

print("X_train_full:", X_train_full.shape, X_train_full.dtype)
print("y_train_full:", y_train_full.shape, y_train_full.dtype)
print("X_test      :", X_test.shape)

Dues coses per mirar a la sortida.

**`X_train_full` té forma (60000, 784).** Seixanta mil imatges i, per a cadascuna,
784 números. Són els 28 × 28 píxels posats en fila. És el que al 5134 en dieu una
taula de 60.000 mostres i 784 característiques, i vol dir que tot el que sabeu
d'scikit-learn hi funcionarà sense canvis.

**El tipus és `uint8`**: enters de 0 a 255, un byte per píxel. Així el fitxer ocupa
només uns 11 MB.

La partició en entrenament (60.000) i test (10.000) ja ve feta de fàbrica i és la
que fa servir tothom, de manera que els resultats es poden comparar amb els
publicats.

#### Una alternativa: carregar-lo directament des d'scikit-learn

MNIST és tan conegut que moltes biblioteques el porten incorporat. A scikit-learn es
descarrega del repositori públic OpenML amb una sola línia (és el que fa Aurélien
Géron al capítol 3 del seu llibre):

```python
from sklearn.datasets import fetch_openml

mnist = fetch_openml("mnist_784", version=1, as_frame=False)
X, y = mnist.data, mnist.target.astype(int)   # the first 60,000 rows are the train set
```

Keras té `keras.datasets.mnist.load_data()` i PyTorch té `torchvision.datasets.MNIST`.
Nosaltres fem servir una còpia al repositori del mòdul per una qüestió pràctica: cada
sessió nova de Colab hauria de tornar a descarregar-lo d'OpenML, cosa que pot tardar
més d'un minut, i no volem dependre d'un servidor extern el dia de classe. **Les
dades són exactament les mateixes**: la còpia es va generar amb `fetch_openml`.

### 1.2 Mirar les dades abans de tocar-les

Com sempre al 5134: abans d'entrenar res, mirem què tenim. Una fila de 784 números
no ens diu res; hem de tornar-la a la seva forma de 28 × 28.

In [ ]:
first_image = X_train_full[0].reshape(28, 28)

print("Label:", y_train_full[0])
print()
# Print the central rows as numbers: 0 is white paper, 255 is full ink
for row in first_image[4:24]:
    print(" ".join(f"{value:3d}" for value in row[4:24]))

Això és el que «veu» el model: una graella de números. Si mires els valors alts, hi
pots endevinar la forma d'un 5. El 0 és paper en blanc i el 255 és tinta.

Ara dibuixem-ne unes quantes com a imatges.

In [ ]:
fig, axes = plt.subplots(3, 10, figsize=(12, 4))
for ax, image, label in zip(axes.ravel(), X_train_full, y_train_full):
    ax.imshow(image.reshape(28, 28), cmap="gray_r")
    ax.set_title(label)
    ax.axis("off")
plt.tight_layout()
plt.show()

Hi ha molta varietat: sets amb barra i sense, uns que semblen quatres, zeros quasi
tancats. Aquesta variació és el que fa el problema interessant i el que fa impossible
resoldre'l amb regles escrites a mà.

Mirem ara quantes imatges hi ha de cada dígit.

In [ ]:
counts = pd.Series(y_train_full).value_counts().sort_index()
counts.plot(kind="bar", color="tab:blue")
plt.xlabel("digit")
plt.ylabel("images")
plt.title("Training images per digit")
plt.xticks(rotation=0)
plt.show()

print(f"Most frequent: {counts.idxmax()} ({counts.max()}), least frequent: {counts.idxmin()} ({counts.min()})")

Les deu classes estan **aproximadament equilibrades**: entre unes 5.400 i unes 6.700
imatges cadascuna. Això vol dir que l'encert (*accuracy*) serà una mètrica raonable
per al problema complet. Aviat veurem un cas en què no ho és.

### 1.3 Escalar, i separar la validació

Al NB 1.1 vam veure que **a les xarxes neuronals escalar les entrades no és
opcional**. Amb els pingüins fèiem l'estandardització (restar la mitjana i dividir per
la desviació). Aquí farem una cosa més simple: **dividir per 255**, de manera que
tots els píxels vagin de 0 a 1.

Per què no estandarditzar? Perquè els píxels **ja estan tots a la mateixa escala**, i
perquè hi ha píxels de les vores que valen 0 a totes les imatges: la seva desviació
típica és 0 i dividir per zero ens donaria un problema.

A més, **separem 10.000 imatges d'entrenament com a validació**. El test el deixem
**segellat fins al final** del notebook, igual que feu amb el test de la UT2 del 5134:
les decisions (quantes neurones, quantes èpoques, quin model) les prendrem mirant la
validació. Si les prenguéssim mirant el test, el resultat final seria massa optimista.

In [ ]:
X_train_full = X_train_full.astype(np.float32) / 255.0
X_test = X_test.astype(np.float32) / 255.0

# The original MNIST train set is already shuffled, so the last 10,000 images are a fair validation set
X_train, X_val = X_train_full[:50_000], X_train_full[50_000:]
y_train, y_val = y_train_full[:50_000], y_train_full[50_000:]

print("Train     :", X_train.shape, "pixel range", X_train.min(), "-", X_train.max())
print("Validation:", X_val.shape)
print("Test      :", X_test.shape, "(sealed until section 9)")

Hem convertit a `float32` en lloc del `float64` per defecte de NumPy. Ocupa la meitat
de memòria (50.000 × 784 números són 39 milions de valors). Les biblioteques de xarxes
neuronals fan servir `float32` per defecte perquè, a més, els càlculs van més ràpid, i
`MLPClassifier` de scikit-learn hi treballa directament.

### 1.4 El model de referència

Abans de cap xarxa, la pregunta de sempre: **quin encert tindria un model que no
aprèn res?**

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy accuracy on validation: {dummy.score(X_val, y_val):.3f}")

Un 11%, gairebé el mateix que triar a l'atzar entre deu dígits. Qualsevol model que
entrenem ha de quedar molt per sobre d'aquest número.

## 2. Una sola neurona: «és un zero?»

Comencem amb el que ja coneixem: el `Perceptron` de scikit-learn del NB 1.1. Com que
només sap distingir dues classes, li plantegem una pregunta de sí o no: **aquesta
imatge és un 0?**

Com al NB 1.1, l'entrenem **època a època** amb `partial_fit` per veure quantes
imatges d'entrenament encara classifica malament després de cada passada. Colab no
recorda res d'un notebook a l'altre, així que tornem a definir la funció auxiliar
`train_by_epoch`.

In [ ]:
def train_by_epoch(model, X, y, n_epochs):
    """Train `model` one epoch at a time with partial_fit; return the misclassified samples per epoch."""
    errors = []
    for _ in range(n_epochs):
        model.partial_fit(X, y, classes=np.array([0, 1]))
        errors.append(int(np.sum(model.predict(X) != y)))
    return errors


y_train_is_zero = (y_train == 0).astype(int)
y_val_is_zero = (y_val == 0).astype(int)

start = time.perf_counter()
perceptron_zero = Perceptron(eta0=0.1, shuffle=False, random_state=42)
errors_zero = train_by_epoch(perceptron_zero, X_train, y_train_is_zero, n_epochs=5)
print(f"Trained in {time.perf_counter() - start:.1f} s")
print("Errors per epoch:", errors_zero)
print(f"Validation accuracy: {perceptron_zero.score(X_val, y_val_is_zero):.3f}")

Els errors **no arriben mai a zero**, i ni tan sols baixen de manera ordenada: d'una
època a l'altra pugen i baixen, entre uns 700 i quasi 2.000 imatges mal classificades.
Recorda el teorema de convergència del NB 1.1: el perceptró només s'estabilitza si les
classes es poden separar amb un hiperplà. Amb 50.000 zeros i no-zeros escrits per
persones diferents, no es pot, i les correccions no s'acaben mai.

Fixa't també en el temps: menys d'un segon per a cinc èpoques de 50.000 imatges.
scikit-learn fa els càlculs amb codi compilat, no amb bucles de Python.

L'encert sembla molt bo. Però abans de celebrar-ho, la pregunta de sempre al 5134:
**comparat amb què?**

In [ ]:
from sklearn.metrics import precision_score, recall_score

pred_zero = perceptron_zero.predict(X_val)
always_no = np.zeros_like(y_val_is_zero)

print(f"Share of zeros in validation      : {y_val_is_zero.mean():.3f}")
print(f"Accuracy of 'never a zero'        : {np.mean(always_no == y_val_is_zero):.3f}")
print(f"Accuracy of the perceptron        : {np.mean(pred_zero == y_val_is_zero):.3f}")
print()
print(f"Recall    (zeros it finds)        : {recall_score(y_val_is_zero, pred_zero):.3f}")
print(f"Precision (its 'zero' calls right): {precision_score(y_val_is_zero, pred_zero):.3f}")

Aquí hi ha la trampa que veureu a fons a la **UT4 del 5134**. Només un 10% de les
imatges són zeros, així que un model que respon sempre «no és un zero» ja encerta un
90%. El perceptró supera aquest número, però per molt menys del que semblava.

Les altres dues mètriques expliquen millor què passa:

- El **recall** diu quina part dels zeros reals troba. És molt alt: gairebé no se
  n'escapa cap.
- La **precisió** diu quina part de les vegades que diu «zero» té raó. És més baixa:
  també marca com a zero moltes imatges que no ho són.

Per a aquest problema **el perceptró ha estat massa generós**: troba els zeros, però
se n'inventa molts. Mirem què ha après.

### 2.1 Els pesos d'una neurona són una imatge

El perceptró té 784 pesos, un per píxel. Si els tornem a posar en una graella de
28 × 28, veiem **quins píxels li fan pensar que és un zero** (pes positiu, vermell) i
**quins li fan pensar que no ho és** (pes negatiu, blau).

In [ ]:
def show_weights(weights, ax=None, title=None):
    """Draw a 784-weight vector as a 28x28 image: red = positive, blue = negative."""
    ax = ax or plt.gca()
    image = np.asarray(weights).reshape(28, 28)
    limit = np.abs(image).max()
    ax.imshow(image, cmap="RdBu_r", vmin=-limit, vmax=limit)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    if title:
        ax.set_title(title)


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
show_weights(perceptron_zero.coef_.ravel(), ax=ax1, title="Perceptron weights: 'is it a 0?'")
ax2.imshow(X_train[y_train == 0].mean(axis=0).reshape(28, 28), cmap="gray_r")
ax2.set_title("Average of all training zeros")
ax2.axis("off")
plt.tight_layout()
plt.show()

A la dreta, la mitjana de tots els zeros d'entrenament: un anell borrós. A l'esquerra,
els pesos del perceptró. Siguem sincers: **és una imatge sorollosa i costa de llegir**.
El que més s'hi veu és on **resta**: una taca blava vertical al centre, just on un 1 o
un 7 tenen tinta i un zero no en té, i una franja blava a baix.

El soroll ve de la regla del perceptró. Com que les classes no es poden separar,
cada època acaba amb centenars de correccions mostra a mostra, i els pesos queden on
els ha deixat l'última. A la secció 6.3 veurem pesos d'un model lineal molt més nets, i
entendrem què els fa diferents.

La idea que convé retenir ara és que **una neurona compara la imatge amb un patró fix
de píxels**. Un zero petit, inclinat o desplaçat no hi encaixa bé.

## 3. De 2 classes a 10: la funció softmax

Per reconèixer els deu dígits alhora necessitem **deu neurones de sortida**, una per
dígit. Cadascuna dona una puntuació, i volem convertir aquestes deu puntuacions en
**deu probabilitats que sumin 1**.

La sigmoide no serveix, perquè tracta cada neurona per separat i les deu
probabilitats podrien sumar qualsevol cosa. La funció que ho fa és la **softmax**, i
fa dues coses:

1. Converteix cada puntuació en un número positiu, de manera que les puntuacions
   altes destaquin **molt** més que les baixes.
2. Divideix cada valor per la suma de tots, perquè sumin 1.

No cal programar-la: és a `scipy.special.softmax`, i vegem què fa amb deu puntuacions
inventades.

In [ ]:
from scipy.special import softmax

scores = np.array([1.0, 2.0, 0.5, 6.0, 1.5, 4.0, 0.0, 2.5, 3.0, 1.0])   # raw outputs for digits 0..9
probabilities = softmax(scores)

for digit, (s, p) in enumerate(zip(scores, probabilities)):
    print(f"digit {digit}: score {s:4.1f}  ->  probability {p:.3f}  {'#' * int(p * 50)}")
print(f"\nSum of probabilities: {probabilities.sum():.3f}   Predicted digit: {probabilities.argmax()}")

El dígit 3 tenia la puntuació més alta (6) i s'emporta la major part de la
probabilitat. El 5, amb un 4, en rep una part apreciable. La resta, gairebé res. La
predicció final és el dígit amb més probabilitat: `argmax`.

### 3.1 Les etiquetes en format *one-hot*

Per comparar deu probabilitats amb la resposta correcta, la resposta també ha de ser
un vector de deu posicions: tot zeros menys un 1 a la posició del dígit correcte. En
diem codificació **one-hot**, i és la mateixa que veureu a la UT3 del 5134 per a les
variables categòriques. scikit-learn té `LabelBinarizer` per fer-ho:

In [ ]:
from sklearn.preprocessing import LabelBinarizer

print("Labels :", y_train[:5])
print(LabelBinarizer().fit(y_train).transform(y_train[:5]))

### 3.2 Què canvia per a la xarxa? Res que hàgim de fer nosaltres

La pèrdua per a deu classes és l'**entropia creuada categòrica**: mira quina
probabilitat ha donat la xarxa **a la classe correcta**. Si és 0,99, la pèrdua és
gairebé 0; si és 0,01, es dispara. És la mateixa idea que l'entropia creuada binària
del NB 1.1, i `log_loss` la calcula igual per a dues classes que per a deu.

La bona notícia és que **`MLPClassifier` ho fa tot sol**. Quan li donem etiquetes amb
més de dues classes, posa deu neurones a la sortida, hi aplica la softmax, converteix
les etiquetes a one-hot per dins i fa servir l'entropia creuada categòrica. La
retropropagació és exactament la mateixa: repartir la culpa cap enrere, capa a capa.
Ho comprovarem a l'atribut `out_activation_` quan tinguem la xarxa entrenada.

## 4. Mini-lots

Al NB 1.1, amb quatre punts de la XOR, cada passa de l'entrenament feia servir
**totes les mostres**. Amb 50.000 imatges, cada passa voldria dir processar-les totes
abans de moure un sol pes. Seria com decidir cap a on baixar la muntanya després
d'enquestar tota la població.

La solució és el **descens del gradient per mini-lots** (*mini-batch gradient
descent*):

1. A cada època, **barregem** les mostres.
2. Les partim en **lots** petits, per exemple de 64 imatges.
3. Per a **cada lot**, calculem cap a on s'han de moure els pesos i fem una passa.

La direcció que dona un lot de 64 és una **estimació** amb soroll de la que donarien
totes les imatges, però és prou bona per anar pel bon camí, i ens permet fer **782
passes per època** en lloc d'una. Dues paraules que convé no confondre:

- Una **iteració** és una passa: un lot processat i els pesos actualitzats.
- Una **època** és una passada per totes les mostres: aquí, 782 iteracions.

La mida del lot és un **hiperparàmetre** més: a `MLPClassifier` es diu `batch_size`.

### 4.1 La xarxa i el seu entrenament

Farem servir `MLPClassifier` amb els paràmetres següents:

- `hidden_layer_sizes=(128,)`: una capa oculta de 128 neurones.
- `activation="tanh"`: la mateixa activació que al NB 1.1.
- `solver="sgd"` i `momentum=0`: el descens del gradient **tal com l'hem explicat**,
  sense millores, amb `learning_rate_init=0.1`.
- `batch_size=64`: la mida de cada mini-lot.

Volem veure com evoluciona l'entrenament **i també la validació** al final de cada
època. Per això, en lloc d'un sol `fit`, farem servir `partial_fit`, que fa **una
època** cada vegada que el crides (barrejant les mostres i recorrent-les per lots), i
després de cada època apuntarem la pèrdua i l'encert a entrenament i a validació.

In [ ]:
def train_with_validation(model, X_train, y_train, X_val, y_val, n_epochs):
    """Train `model` one epoch at a time and record loss and accuracy on train and validation."""
    classes = np.unique(y_train)
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    for _ in range(n_epochs):
        model.partial_fit(X_train, y_train, classes=classes)
        for split, X, y in [("train", X_train, y_train), ("val", X_val, y_val)]:
            history[f"{split}_loss"].append(log_loss(y, model.predict_proba(X), labels=classes))
            history[f"{split}_acc"].append(model.score(X, y))
    return history


def make_mlp(n_hidden=128, batch_size=64, random_state=42):
    """The network of this notebook: one tanh hidden layer trained with plain mini-batch SGD."""
    return MLPClassifier(hidden_layer_sizes=(n_hidden,), activation="tanh", solver="sgd",
                         momentum=0, learning_rate_init=0.1, batch_size=batch_size,
                         random_state=random_state)


def n_parameters(model):
    """Total number of learned weights and biases."""
    return sum(W.size for W in model.coefs_) + sum(b.size for b in model.intercepts_)

Dues funcions petites, perquè les farem servir diverses vegades:

- **`train_with_validation`** entrena època a època i guarda quatre llistes: pèrdua i
  encert, a entrenament i a validació. La pèrdua la mesura `log_loss` a partir de les
  probabilitats de `predict_proba`.
- **`make_mlp`** crea la xarxa amb els paràmetres de sobre. Així, quan canviem la mida
  de la capa o del lot, la resta queda igual.
- **`n_parameters`** suma la mida de totes les matrius de pesos (`coefs_`) i de tots
  els biaixos (`intercepts_`).

## 5. Entrenar la xarxa

Una xarxa **784-128-10**: 784 entrades, 128 neurones ocultes i 10 sortides. Tenim

- $784 \times 128 + 128 = 100.480$ paràmetres a la capa oculta,
- $128 \times 10 + 10 = 1.290$ a la de sortida.

Més de **100.000 pesos** que aprendre. La xarxa de la XOR en tenia 13.

> ⏱ Aquesta cel·la tarda uns segons en un ordinador i pot arribar a un minut a Colab.

In [ ]:
n_epochs = 20

start = time.perf_counter()
mlp = make_mlp()
history = train_with_validation(mlp, X_train, y_train, X_val, y_val, n_epochs)
mlp_fit_time = time.perf_counter() - start

print(f"Parameters       : {n_parameters(mlp):,}")
print(f"Output activation: {mlp.out_activation_}")
print(f"Trained in {mlp_fit_time:.1f} s")
print(f"Train accuracy     : {history['train_acc'][-1]:.4f}")
print(f"Validation accuracy: {history['val_acc'][-1]:.4f}")

La xarxa reconeix correctament **al voltant del 97,6%** dels dígits de validació, que
no ha vist mai.

I l'atribut `out_activation_` confirma el que dèiem a la secció 3.2: com que les
etiquetes tenen deu classes, `MLPClassifier` ha posat sol la **softmax** a la sortida.
Nosaltres no hem hagut de canviar res respecte del NB 1.1.

### 5.1 Les corbes d'aprenentatge

Al NB 1.1 miràvem la corba de pèrdua d'entrenament (`loss_curve_`). Ara que tenim
validació, dibuixem les dues, perquè és comparant-les que es veu si la xarxa aprèn o
memoritza.

In [ ]:
epochs = np.arange(1, n_epochs + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, history["train_loss"], "o-", label="train")
ax1.plot(epochs, history["val_loss"], "o-", label="validation")
ax1.set_xlabel("epoch")
ax1.set_ylabel("cross-entropy")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs, history["train_acc"], "o-", label="train")
ax2.plot(epochs, history["val_acc"], "o-", label="validation")
ax2.set_xlabel("epoch")
ax2.set_ylabel("accuracy")
ax2.set_title("Accuracy")
ax2.legend()
plt.tight_layout()
plt.show()

Llegeix les dues corbes alhora:

- Durant les primeres èpoques, **entrenament i validació milloren junts**. La xarxa
  aprèn coses que serveixen per a dígits nous.
- Cap al final, **l'entrenament continua millorant però la validació gairebé s'atura**
  i les corbes se separen. La xarxa comença a aprendre detalls de les 50.000 imatges
  concretes que no es generalitzen: és l'inici del **sobreajust** (UT2 del 5134).

Si continuéssim moltes èpoques més, la pèrdua de validació acabaria pujant. Decidir
en quina època ens quedem amb el model és exactament el que la UT10 del 5134
anomena «decidir en quina iteració es desa la versió del model». En tens un exercici.

### 5.2 Per què mini-lots? Un experiment

Entrenem la mateixa xarxa **5 èpoques** amb tres mides de lot i comparem l'encert de
validació al final de cada època. Totes tres veuen **exactament les mateixes dades el
mateix nombre de vegades**.

> ⏱ Tres entrenaments curts: uns segons, o prop d'un minut a Colab.

In [ ]:
batch_results = {}
for batch_size in [50_000, 1024, 64]:
    start = time.perf_counter()
    model = make_mlp(batch_size=batch_size)
    run = train_with_validation(model, X_train, y_train, X_val, y_val, n_epochs=5)
    batch_results[batch_size] = (run["val_acc"], time.perf_counter() - start)

for batch_size, (val_acc, seconds) in batch_results.items():
    steps = int(np.ceil(50_000 / batch_size)) * 5
    label = "full batch" if batch_size == 50_000 else f"batch {batch_size}"
    plt.plot(range(1, 6), val_acc, "o-", label=f"{label}: {steps:,} steps, {seconds:.1f} s")

plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Same data, same epochs, different batch sizes")
plt.xticks(range(1, 6))
plt.legend()
plt.show()

El resultat és contundent:

- **Lot complet** (50.000 imatges, el que fèiem al NB 1.1): 5 èpoques són només
  **5 passes**. Després de passar cinc vegades per totes les dades, la xarxa encara
  va perduda.
- **Lots de 1.024**: 245 passes. Molt millor: ja passa del 90%.
- **Lots de 64**: 3.910 passes, i supera el 95% a la tercera època.

I el temps **no creix al mateix ritme que les passes**: amb lots de 64 es fan quasi
800 vegades més passes que amb el lot complet, però el temps només es multiplica per
tres o quatre. Processar 50.000 imatges costa gairebé el mateix tant si les agrupes en
un lot com en 782; la diferència és quantes vegades aprofites aquest càlcul per
corregir els pesos.

Per què no fer lots d'1 imatge, llavors? Perquè el gradient d'una sola imatge té
moltíssim soroll i perquè els ordinadors (i sobretot les GPU) són eficients multiplicant matrius grans,
no fent milers de càlculs petits. Els valors habituals estan entre 32 i 512.

### 5.3 Quantes neurones ocultes?

Com al NB 1.1, però ara amb dades de veritat.

> ⏱ Quatre entrenaments de 10 èpoques. El de 256 neurones és el més lent. A Colab,
> compta un parell de minuts.

In [ ]:
rows = []
for n_hidden in [4, 16, 64, 256]:
    start = time.perf_counter()
    model = make_mlp(n_hidden=n_hidden)
    run = train_with_validation(model, X_train, y_train, X_val, y_val, n_epochs=10)
    rows.append({"hidden neurons": n_hidden,
                 "parameters": n_parameters(model),
                 "train accuracy": run["train_acc"][-1],
                 "validation accuracy": run["val_acc"][-1],
                 "fit time (s)": time.perf_counter() - start})

pd.DataFrame(rows).set_index("hidden neurons").round(4)

- Amb **4 neurones** la xarxa ha d'explicar deu dígits amb només quatre
  característiques, i es queda molt curta.
- De **4 a 16** el salt és enorme, i de **16 a 64** encara es guanyen uns dos punts.
- De **64 a 256** l'encert gairebé no millora, però els paràmetres es multipliquen per
  quatre i el temps, per tres.

És un patró que trobareu sovint: **a partir d'una mida, afegir capacitat surt car i
aporta poc**. Quina mida triar és una decisió d'enginyeria, no de precisió.

## 6. Què ha après la capa oculta?

Aquesta és la secció més important del notebook, perquè tanca el fil que vam obrir al
NB 1.1: **als algorismes clàssics les característiques les prepara la persona; a les
xarxes, les capes ocultes les aprenen.** Amb la XOR ho vam veure en un pla. Ara ho
veurem amb imatges i ho **mesurarem**.

### 6.1 Els pesos de les neurones ocultes

Cada neurona oculta té 784 pesos, un per píxel. Com amb el perceptró de la secció 2,
els podem dibuixar com una imatge de 28 × 28.

In [ ]:
fig, axes = plt.subplots(4, 8, figsize=(12, 6.5))
for neuron, ax in enumerate(axes.ravel()):
    show_weights(mlp.coefs_[0][:, neuron], ax=ax, title=f"neuron {neuron}")
fig.suptitle("What 32 of the 128 hidden neurons respond to (red +, blue -)", fontsize=13)
plt.tight_layout()
plt.show()

Compara aquestes imatges amb la plantilla del perceptró de la secció 2. Aquí **cap
neurona no s'assembla a un dígit sencer**. El que hi veiem són **traços**: corbes en
una zona concreta, línies inclinades, una taca vermella al costat d'una de blava
(«tinta aquí però no allà»). Algunes neurones semblen soroll: no totes aprenen coses
fàcils d'interpretar.

Cada neurona detecta **un fragment**, i la capa de sortida decideix el dígit combinant
quins fragments hi ha. Un 7 és «traç horitzontal a dalt + diagonal llarga»; un 4 i un
9 comparteixen traços, i per això es confonen.

**Ningú no ha dit a la xarxa que busqués traços.** Només li hem donat píxels i
etiquetes.

### 6.2 La prova: característiques apreses contra píxels

Una imatge suggereix, però no demostra. Fem un experiment que ho **mesuri**, amb una
eina que coneixeu del 5134: la **regressió logística**, un model lineal que no pot
crear característiques noves.

L'entrenem tres vegades, canviant només què li donem com a entrada:

1. Els **784 píxels** originals.
2. Les **128 activacions de la capa oculta** de la xarxa entrenada: el que «veu» la
   capa de sortida.
3. Les **128 activacions d'una xarxa amb pesos aleatoris**, sense entrenar. És el
   control: comprova si la millora ve de l'aprenentatge o simplement de passar les
   dades per 128 neurones qualssevol.

Per obtenir les activacions de la capa oculta fem el mateix que al NB 1.1: suma
ponderada amb els pesos de la primera capa (`coefs_[0]` i `intercepts_[0]`) i després
la tanh. Per a la xarxa sense entrenar, scikit-learn no crea els pesos fins que no
comença a entrenar, així que li fem fer **una sola passa amb un coeficient
d'aprenentatge minúscul**: els pesos queden creats, però pràcticament intactes.

> ⏱ La primera regressió, amb 784 columnes, tarda uns segons; les altres dues són molt més ràpides.

In [ ]:
def hidden_activations(model, X):
    """What the output layer 'sees': the values of the first hidden layer."""
    return np.tanh(X @ model.coefs_[0] + model.intercepts_[0])


# One tiny step only creates the (random) weights, it does not learn anything
untrained = MLPClassifier(hidden_layer_sizes=(128,), activation="tanh", solver="sgd", momentum=0,
                          learning_rate_init=1e-12, batch_size=64, random_state=42)
untrained.partial_fit(X_train[:64], y_train[:64], classes=np.arange(10))

inputs = {
    "784 raw pixels": (X_train, X_val),
    "128 learned hidden features": (hidden_activations(mlp, X_train), hidden_activations(mlp, X_val)),
    "128 random hidden features": (hidden_activations(untrained, X_train), hidden_activations(untrained, X_val)),
}

rows = []
for name, (A_train, A_val) in inputs.items():
    start = time.perf_counter()
    logreg = LogisticRegression(max_iter=1000).fit(A_train, y_train)
    rows.append({"input to the logistic regression": name,
                 "columns": A_train.shape[1],
                 "validation accuracy": logreg.score(A_val, y_val),
                 "fit time (s)": time.perf_counter() - start})
    if name == "784 raw pixels":
        logreg_pixels = logreg

pd.DataFrame(rows).set_index("input to the logistic regression").round(4)

Aquesta taula és el resum de tota la unitat:

- Amb els **784 píxels**, la regressió logística es queda per sota del 93%. Una
  frontera lineal sobre píxels no dona per més.
- Amb les **128 característiques apreses**, **el mateix model lineal** salta fins a
  prop del 98%, amb **sis vegades menys columnes** i entrenant-se en molt menys temps.
- Amb **128 característiques aleatòries**, queda **pitjor que amb els píxels**. Passar
  les dades per neurones no fa màgia: el que val és que **els pesos s'hagin après**.

És la versió a gran escala de la figura de la XOR al NB 1.1: la capa oculta
**transforma les dades** en una representació on el problema es resol amb una frontera
lineal. Allà ho vam fer amb 4 punts i 2 neurones triades a mà. Aquí ho ha fet
l'entrenament, amb 50.000 imatges i 128 neurones.

Al 5134, a la UT3, preparareu característiques noves a mà, i a la UT8 reduireu la
dimensió de les dades amb PCA. La capa oculta fa les dues coses alhora, i guiada per
l'objectiu: passa de 784 columnes a 128, i les tria **perquè serveixin per distingir
dígits**.

### 6.3 Plantilles contra traços

Per acabar de veure la diferència, dibuixem els pesos que aprèn la regressió logística
sobre els píxels: un vector de 784 pesos **per a cada dígit**.

Amb la configuració per defecte els pesos surten plens de soroll, com els del
perceptró: cada píxel té un pes lliure i el model n'aprofita qualsevol detall. Per
veure'ls clars n'entrenem una versió amb **més regularització**: el paràmetre `C` de la
UT4 del 5134. Un `C` petit castiga els pesos grans i obliga el model a fer servir
patrons suaus. Perd poc encert i guanya molta llegibilitat.

In [ ]:
logreg_smooth = LogisticRegression(C=0.01, max_iter=1000).fit(X_train, y_train)
print(f"Validation accuracy with C=1 (default): {logreg_pixels.score(X_val, y_val):.4f}")
print(f"Validation accuracy with C=0.01       : {logreg_smooth.score(X_val, y_val):.4f}")

fig, axes = plt.subplots(2, 10, figsize=(15, 3.6))
for digit in range(10):
    show_weights(logreg_pixels.coef_[digit], ax=axes[0, digit], title=f"class {digit}")
    show_weights(logreg_smooth.coef_[digit], ax=axes[1, digit])
axes[0, 0].set_ylabel("C = 1")
axes[1, 0].set_ylabel("C = 0.01")
fig.suptitle("Logistic regression on pixels: one template per digit", fontsize=13)
plt.tight_layout()
plt.show()

A la fila de dalt, amb `C = 1`, costa veure-hi res. A la de baix, amb més
regularització, **s'hi reconeixen els dígits**: el 0 és un anell vermell amb el centre
blau, l'1 una barra vertical, el 3 i el 6 les seves corbes. Un model lineal **només pot
aprendre una plantilla per classe**, i cada imatge es compara amb les deu plantilles.

La xarxa, en canvi, ha après **128 peces reutilitzables** i les combina. Això és el que
fa que una sola capa oculta passi del 93% a més del 97%.

## 7. Quins errors fa?

Un encert del 97,6% vol dir 235 dígits de validació mal classificats. Abans del
número global, com diem sempre al 5134, **mirem casos concrets**.

### 7.1 La matriu de confusió

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

val_pred = mlp.predict(X_val)
print(f"Misclassified validation images: {np.sum(val_pred != y_val)} of {len(y_val)}")

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, ax=ax, colorbar=False)
ax.grid(False)
ax.set_title("Confusion matrix on validation")
plt.show()

errors = confusion_matrix(y_val, val_pred)
np.fill_diagonal(errors, 0)
worst = np.dstack(np.unravel_index(np.argsort(errors.ravel())[::-1][:3], errors.shape))[0]
for true_digit, predicted in worst:
    print(f"  {errors[true_digit, predicted]:3d} times a {true_digit} was taken for a {predicted}")

Les files són el dígit real i les columnes, el predit. La diagonal són els encerts, i
tot el que queda fora són errors. És la mateixa matriu de la UT4 del 5134, però amb deu
classes en lloc de dues.

Les confusions més freqüents no són a l'atzar: **5 pres per 6**, **5 pres per 3** i
**9 pres per 4**. Són parelles de dígits que comparteixen traços (la panxa de baix del
5, del 6 i del 3; el pal i el tancament del 4 i el 9), exactament com suggerien les
imatges de la capa oculta.

### 7.2 Mirar els errors

In [ ]:
val_proba = mlp.predict_proba(X_val)
wrong = np.where(val_pred != y_val)[0]
# The 15 errors the network was most confident about
most_confident_wrong = wrong[np.argsort(val_proba[wrong].max(axis=1))[::-1][:15]]

fig, axes = plt.subplots(3, 5, figsize=(10, 7))
for ax, i in zip(axes.ravel(), most_confident_wrong):
    ax.imshow(X_val[i].reshape(28, 28), cmap="gray_r")
    ax.set_title(f"true {y_val[i]} | pred {val_pred[i]} ({val_proba[i].max():.0%})", fontsize=10)
    ax.axis("off")
fig.suptitle("Errors made with the highest confidence", fontsize=13)
plt.tight_layout()
plt.show()

Hem triat els errors en què la xarxa estava **més segura**. Mira-los amb calma: n'hi ha
uns quants que **tu mateix no sabries classificar**, dígits mal escrits o que
s'assemblen més a un altre número que al que diu l'etiqueta. D'altres, en canvi, són
errors clars que una persona no faria.

Això connecta amb una idea que treballareu a la UT4 del 5134: `predict_proba` ens dona
**la confiança** del model. En una aplicació real (llegir codis postals, xecs,
formularis) es pot establir un llindar: si la probabilitat màxima és baixa, el sistema
**no decideix** i passa la imatge a una persona.

## 8. Comparació amb els algorismes clàssics

Ara la pregunta que connecta els dos mòduls. **Una xarxa neuronal és millor que els
algorismes clàssics amb aquestes dades?** Posem-los tots en una taula.

- `DummyClassifier`: la referència.
- `LogisticRegression`: model lineal (UT4 del 5134). Ja l'hem entrenat a la secció 6.
- `RandomForestClassifier`: bosc aleatori de 100 arbres (UT5 del 5134).
- La xarxa de la secció 5: tanh i descens del gradient pur.
- `MLPClassifier` **amb els valors per defecte** de scikit-learn i la mateixa mida de
  capa: activació ReLU i l'optimitzador Adam, 20 èpoques.

> ⏱ El bosc aleatori i l'`MLPClassifier` tarden una estona. A Colab, un parell de minuts en total.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

sklearn_models = {
    "Random forest (100 trees)": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42),
    "MLPClassifier defaults (128, relu, adam)": MLPClassifier(hidden_layer_sizes=(128,), max_iter=20, random_state=42),
}

results = [
    {"model": "Dummy (reference)", "validation accuracy": dummy.score(X_val, y_val), "fit time (s)": 0.0},
    {"model": "Logistic regression", "validation accuracy": logreg_pixels.score(X_val, y_val),
     "fit time (s)": rows[0]["fit time (s)"]},
    {"model": "MLP of section 5 (128, tanh, sgd)", "validation accuracy": mlp.score(X_val, y_val),
     "fit time (s)": mlp_fit_time},
]

for name, model in sklearn_models.items():
    start = time.perf_counter()
    with warnings.catch_warnings():
        # MLPClassifier warns that 20 epochs are not enough to fully converge: we stop it on purpose
        warnings.simplefilter("ignore", category=ConvergenceWarning)
        model.fit(X_train, y_train)
    results.append({"model": name, "validation accuracy": model.score(X_val, y_val),
                    "fit time (s)": time.perf_counter() - start})

comparison = pd.DataFrame(results).set_index("model").sort_values("validation accuracy")
comparison.round(4)

Llegeix la taula amb honestedat, perquè no diu el que potser esperaves.

**Les xarxes guanyen, però el bosc aleatori queda molt a prop.** Menys d'un punt de
diferència. Un algorisme clàssic, sense cap capa oculta, aconsegueix un resultat quasi
igual.

**La regressió logística sí que queda clarament per sota**, perquè és lineal. Això és
coherent amb tot el que hem vist: el que marca la diferència és poder fer fronteres
no lineals, i el bosc aleatori també en sap fer.

**Les dues xarxes obtenen resultats molt semblants.** La configuració per defecte de
scikit-learn, amb ReLU i l'optimitzador Adam, queda una mica per sobre i s'entrena més
de pressa. El temps de la xarxa de la secció 5, a més, inclou avaluar entrenament i
validació al final de cada època per dibuixar les corbes. La idea de fons és la
mateixa: pas endavant, pèrdua, pas enrere i actualització, lot a lot.

**Els temps depenen molt de la màquina**, i sorprenen. Aquí la regressió logística no
és la més ràpida: amb 784 columnes i 50.000 files, el seu optimitzador necessita
centenars d'iteracions. Al NB 1.1, amb dues columnes, s'entrenava en mil·lisegons. A
Colab, a més, el bosc aleatori pot tardar bastant més perquè té menys nuclis per
repartir els arbres.

Llavors, per què han guanyat les xarxes neuronals en imatges? La secció 10 en dona la
pista.

## 9. Obrim el test segellat

Hem pres totes les decisions (128 neurones, 20 èpoques, lots de 64) mirant **només** la
validació. Ara, i només ara, avaluem el model triat sobre les 10.000 imatges de test.

In [ ]:
test_accuracy = mlp.score(X_test, y_test)
print(f"Validation accuracy: {mlp.score(X_val, y_val):.4f}")
print(f"Test accuracy      : {test_accuracy:.4f}")

El resultat de test és molt proper al de validació. Això vol dir que **l'estimació que
fèiem amb la validació era fiable**: no hem fet trampa sense voler ajustant-nos a les
dades amb què mesuràvem. És el que fareu a la UT11 del 5134, quan obrireu el test
reservat des de la UT2.

A la bibliografia, un perceptró multicapa d'una capa oculta sobre MNIST dona resultats
d'aquest ordre. Per baixar de l'1-2% d'error calen altres arquitectures.

## 10. El límit: l'MLP no sap que mira una imatge

Fem un últim experiment, molt senzill. Agafem les imatges de validació i les
**desplacem 3 píxels cap a la dreta**. Per a una persona, és el mateix dígit.

In [ ]:
def shift_right(X, pixels):
    """Move every 28x28 image `pixels` columns to the right, filling the left side with blank paper."""
    images = X.reshape(-1, 28, 28)
    shifted = np.zeros_like(images)
    shifted[:, :, pixels:] = images[:, :, :-pixels]
    return shifted.reshape(-1, 784)


X_val_shifted = shift_right(X_val, pixels=3)

fig, axes = plt.subplots(2, 6, figsize=(10, 3.6))
for col in range(6):
    axes[0, col].imshow(X_val[col].reshape(28, 28), cmap="gray_r")
    axes[1, col].imshow(X_val_shifted[col].reshape(28, 28), cmap="gray_r")
    axes[0, col].axis("off")
    axes[1, col].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("shifted 3 px", loc="left")
plt.tight_layout()
plt.show()

shift_rows = []
for name, model in [("Logistic regression", logreg_pixels),
                    ("Random forest (100 trees)", sklearn_models["Random forest (100 trees)"]),
                    ("MLP of section 5 (128, tanh, sgd)", mlp)]:
    shift_rows.append({"model": name,
                       "original validation": model.score(X_val, y_val),
                       "shifted 3 pixels": model.score(X_val_shifted, y_val)})
pd.DataFrame(shift_rows).set_index("model").round(3)

**Tots els models s'enfonsen**, la xarxa inclosa. Dígits que reconeixia amb un 97%
d'encert, desplaçats tres píxels, els falla gairebé la meitat de vegades.

El motiu és com li donem la imatge: **una fila de 784 números**. Per a l'MLP, el píxel
150 i el píxel 151 són dues columnes qualssevol, igual que el píxel 150 i el 700. **No
sap que són veïns**, ni que la mateixa forma pot aparèixer en llocs diferents. Els seus
detectors de traços de la secció 6 funcionen **en una posició fixa**: si el traç es
mou, el detector no el veu.

Això explica la taula de la secció 8. Un perceptró multicapa tracta la imatge com una
taula de dades, igual que un bosc aleatori, i per això els resultats s'assemblen. Les
xarxes que van revolucionar la visió per computador el 2012 són les **xarxes
convolucionals**, que **reutilitzen el mateix detector a totes les posicions** de la
imatge. Tenen en compte l'estructura de les dades, i és llavors quan la distància amb
els algorismes clàssics es fa enorme. Són el tema de la **UT6**, on les entrenarem amb
PyTorch sobre aquestes mateixes imatges.

La idea de fons de la unitat es manté, però amb un matís important: **una xarxa
aprèn característiques, i l'arquitectura decideix quina mena de característiques pot
aprendre.**

## 11. Resum

| | NB 1.1 | NB 1.2 |
|---|---|---|
| **Dades** | XOR, pingüins, dues llunes: 2 entrades | MNIST: 784 entrades, 60.000 imatges |
| **Sortida** | 1 neurona, sigmoide | 10 neurones, softmax |
| **Pèrdua** | Entropia creuada binària | Entropia creuada categòrica |
| **Entrenament** | `fit`, i `partial_fit` per mirar-lo pas a pas | `partial_fit` època a època, amb mini-lots |
| **Avaluació** | Entrenament i test | Entrenament, validació i test segellat |
| **Retropropagació** | La fa `MLPClassifier` | La mateixa: `MLPClassifier` s'adapta sol a deu classes |

### Les idees que t'has d'endur

1. **Una imatge és una fila de números** per a un MLP. Funciona, però es perd la
   informació de quins píxels són veïns.
2. **Softmax** converteix deu puntuacions en deu probabilitats. `MLPClassifier` la posa
   sola quan hi ha més de dues classes (`out_activation_`), i la retropropagació no
   canvia.
3. **Els mini-lots** fan moltes passes per època amb un cost de càlcul semblant. Una
   iteració és un lot; una època, totes les dades. A scikit-learn, `batch_size`.
4. **La capa oculta aprèn característiques**, i ho hem mesurat: un model lineal passa
   del 93% a prop del 98% quan li donem les 128 característiques apreses en lloc dels
   784 píxels.
5. **Amb dades tabulars, un bosc aleatori s'hi acosta molt.** La gran avantatja de les
   xarxes arriba quan l'arquitectura aprofita l'estructura de les dades.

## Exercicis

**Exercici 1. Quan aturar l'entrenament.** Entrena la xarxa de la secció 5 durant 40
èpoques amb `train_with_validation` i dibuixa les corbes de pèrdua de validació i
d'entrenament. En quina època la pèrdua de validació és mínima? Després prova
`MLPClassifier` amb `early_stopping=True`, `validation_fraction=0.1` i
`n_iter_no_change=3`: en quina època s'atura i quin encert de validació obté?

**Exercici 2. ReLU.** Crea la xarxa amb `activation="relu"` en lloc de `"tanh"` i
compara'n les corbes de validació. Potser hauràs de baixar el coeficient
d'aprenentatge (`learning_rate_init`).

**Exercici 3. El coeficient d'aprenentatge.** Entrena 10 èpoques amb
`learning_rate_init` de 0,01, 0,1, 0,5 i 2. Dibuixa l'encert de validació per època
dels quatre. Relaciona el resultat amb les quatre valls de la secció 8 del NB 1.1.

**Exercici 4. Barrejar els píxels.** Tria una permutació fixa dels 784 píxels
(`perm = np.random.default_rng(0).permutation(784)`) i aplica-la a totes les imatges:
`X_train[:, perm]`. Dibuixa'n unes quantes: ja no s'hi reconeix res. Entrena-hi la
xarxa i el bosc aleatori. Què passa amb l'encert? Què et diu això sobre el que l'MLP
«sap» de les imatges? Relaciona-ho amb la secció 10.

**Exercici 5. Aprendre a tolerar desplaçaments.** Crea un conjunt d'entrenament ampliat
amb les 50.000 imatges originals i còpies desplaçades 2 píxels cap a la dreta i cap a
l'esquerra (caldrà una funció `shift_left`). Entrena-hi la xarxa i repeteix
l'experiment de la secció 10. Aquesta tècnica s'anomena **augment de dades** (*data
augmentation*) i la tornarem a veure a la UT6. Quin inconvenient té?

**Exercici 6. Dues capes ocultes.** Entrena una xarxa amb `hidden_layer_sizes=(128, 64)`
i compara-la amb la d'una sola capa: encert de validació, nombre de paràmetres i temps.
Val la pena la segona capa amb aquestes dades?

**Exercici 7. Fashion-MNIST.** Existeix un conjunt amb el mateix format però amb
fotografies de peces de roba: `fetch_openml("Fashion-MNIST", version=1, as_frame=False)`.
Repeteix la taula de comparació de la secció 8. És més fàcil o més difícil que MNIST?
Canvia l'ordre dels models?

**Exercici 8. No decidir quan no n'està segur.** Amb `predict_proba`, calcula quin
percentatge d'imatges de validació té una probabilitat màxima inferior a 0,9, i quin
encert té la xarxa **només** sobre les imatges restants. Dibuixa l'encert en funció del
llindar (de 0,5 a 0,99). Quin llindar triaries per a un sistema que llegeix imports de
xecs bancaris?

## Per al debat de classe

1. **Un bosc aleatori obté gairebé el mateix encert que la nostra xarxa.** Si haguéssiu
   de posar un dels dos en producció per llegir codis postals, quin triaríeu? Quins
   criteris a part de l'encert tindríeu en compte?
2. **L'MLP falla amb dígits desplaçats tres píxels.** Com hauria de ser una xarxa
   neuronal pensada per a imatges? Quina propietat li demanaríeu?
3. **MNIST és de 1998 i es considera un problema resolt.** Per què creieu que encara es
   fa servir per ensenyar i per provar idees noves? Quins perills té avaluar un mètode
   només amb MNIST?
4. **Les persones que van escriure aquests dígits** eren treballadors del cens i
   estudiants. Si el sistema s'hagués de fer servir amb la lletra de persones grans o
   d'un altre país, el 97% es mantindria? Què hauríeu de comprovar?